《Think Stats》 3판은 [Bookshop.org](https://bookshop.org/a/98697/9781098190255)와 [Amazon](https://amzn.to/42lmxwu)에서 구할 수 있습니다. 두 링크는 원저자의 제휴 링크입니다. 책이 도움이 되었다면 Green Tea Press의 [무료 공개 교육 자료 제작을 후원](https://greenteapress.com/support/?book=ts3)하는 것도 생각해 주세요.

# 추정

인구가 1만 명인 마을에 살면서 다가오는 선거의 승자를 예측하려 한다고 하겠습니다. 이론적으로는 모든 주민에게 누구에게 투표할지 물어보고, 모두가 정직하게 답한다면 믿을 만한 예측을 할 수 있습니다.

하지만 작은 마을에서도 모집단 전체를 조사하기는 현실적으로 어려울 것입니다. 다행히 그럴 필요는 없습니다. 무작위로 고른 일부 주민을 조사하면, 그 표본으로 모집단의 투표 선호를 추론할 수 있습니다. 이렇게 표본으로 모집단에 관해 추론하는 과정을 통계적 추론이라고 합니다.

통계적 추론에는 이 장에서 다룰 추정과 다음 장에서 다룰 가설검정이 포함됩니다.

[이 노트북의 원문을 Colab에서 실행하기](https://colab.research.google.com/github/AllenDowney/ThinkStats/blob/v3/nb/chap08.ipynb).

In [1]:
from os.path import basename, exists


def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + local)


download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")

In [2]:
try:
    import empiricaldist
except ImportError:
    %pip install empiricaldist

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from thinkstats import decorate

## 펭귄의 체중 재기

남극에서 지역 펭귄 개체군을 연구하는 연구자라고 하겠습니다. 일 년 동안 펭귄의 평균 체중이 어떻게 변하는지 관찰하는 것이 업무 중 하나입니다. 모든 펭귄의 체중을 재기는 어렵습니다. 그래서 매주 펭귄 10마리를 무작위로 골라 체중을 재고, 그 표본으로 개체군 전체의 평균인 **모평균**(population mean)을 추정하려 합니다.

표본으로 모평균을 추정하는 방법은 많지만, 여기서는 표본평균과 표본중앙값 두 가지만 생각하겠습니다. 둘 다 합리적인 선택입니다. 어느 쪽이 더 나은지, 그리고 “더 낫다”는 것이 무엇인지 살펴보겠습니다.

설명을 위해 펭귄의 체중이 평균과 표준편차를 아는 정규분포에서 나온다고 가정합니다. 평균과 표준편차는 각각 `mu`, `sigma`로 나타내고 킬로그램 단위 값을 지정하겠습니다.

In [4]:
mu = 3.7
sigma = 0.46

이 값들은 정규분포의 **모수**(parameters)입니다. 특정한 분포를 정하는 값이라는 뜻입니다. 모수가 주어지면 NumPy로 표본을 뽑는 과정을 시뮬레이션해 원하는 크기의 표본을 만들 수 있습니다. 다음은 체중 10개로 된 가상 표본입니다.

In [5]:
# Seed the random number generator so we get the same results every time
np.random.seed(1)

In [6]:
sample = np.random.normal(mu, sigma, size=10)
sample

표본의 평균과 중앙값은 다음과 같습니다.

In [7]:
np.mean(sample), np.median(sample)

두 값의 차이가 어느 쪽이 더 나은 추정인지 궁금해질 만큼 큽니다. 알아보기 위해, 주어진 크기 `n`의 가상 표본을 만드는 다음 함수를 사용하겠습니다.

In [8]:
def make_sample(n):
    return np.random.normal(mu, sigma, size=n)

첫 실험으로 표본 크기가 커질 때 표본평균과 표본중앙값이 어떻게 변하는지 보겠습니다. NumPy의 `logspace`로 10부터 100,000까지 로그 눈금에서 등간격인 `ns`를 만듭니다.

In [9]:
ns = np.logspace(1, 5).astype(int)

리스트 컴프리헨션으로 각 `n`마다 가상 표본을 만들고 평균을 구해 결과를 모을 수 있습니다.

In [10]:
means = [np.mean(make_sample(n)) for n in ns]

중앙값에 대해서도 같은 일을 합니다.

In [11]:
medians = [np.median(make_sample(n)) for n in ns]

표본평균이나 표본중앙값처럼 모집단의 성질을 추정하는 데 쓰는 통계량을 **추정량**(estimator)이라고 합니다.

다음 그림은 표본 크기를 늘릴 때 이 추정량들이 어떻게 변하는지 보여 줍니다. 수평선은 실제 모평균입니다.

In [12]:
plt.axhline(mu, color="gray", lw=1, alpha=0.5)
plt.plot(ns, means, "--", label="mean")
plt.plot(ns, medians, alpha=0.5, label="median")

decorate(xlabel="Sample size", xscale="log", ylabel="Estimate")

표본 크기가 커질수록 두 추정량 모두 실제 값 주변에서의 변동이 줄어드는 경향을 보입니다. 좋은 추정량의 성질 중 하나인 **일치성**(consistency)을 보여 주는 실험입니다. 일치성만으로는 둘 중 어느 추정량이 더 나은지 고를 수 없습니다.

그림에서 추정값은 실제 값보다 높을 때도 낮을 때도 있고, 변동은 실제 값을 중심으로 대략 대칭인 듯합니다. 그렇다면 같은 크기의 표본을 많이 뽑아 추정값을 많이 구했을 때, 그 추정값의 평균은 얼마일까요?

다음 반복문은 펭귄 10마리짜리 표본을 10,001번 만들고 각 표본의 평균을 구합니다.

In [13]:
means = [np.mean(make_sample(n=10)) for i in range(10001)]
np.mean(means)

표본평균들의 평균은 표본을 생성할 때 사용한 실제 평균인 3.7kg에 가깝습니다.

다음 반복문은 같은 실험에서 각 표본의 중앙값을 구합니다.

In [14]:
medians = [np.median(make_sample(n=10)) for i in range(10001)]
np.mean(medians)

가상 표본중앙값들의 평균도 실제 모평균에 아주 가깝습니다.

이 정규 모형에서 표본평균과 표본중앙값은 **불편추정량**(unbiased estimators)입니다. 평균적으로 맞는다는 뜻입니다. “편향”이라는 말은 맥락마다 뜻이 달라 혼동하기 쉽습니다. 여기서 “불편”은 추정값들의 평균이 실제 값과 같다는 뜻입니다.

지금까지 두 추정량의 일치성과 불편성을 살펴보았지만, 어느 쪽이 더 좋은지는 아직 분명하지 않습니다. 어느 추정량이 더 정확한지 한 번 더 실험하겠습니다. 추정의 정확도를 비교할 기준으로 **평균제곱오차**(mean squared error, MSE)를 사용하겠습니다. 다음 함수는 추정값과 실제 값의 차이를 구하고 오차 제곱의 평균을 반환합니다.

In [15]:
def mse(estimates, actual):
    """Mean squared error of a sequence of estimates."""
    errors = np.asarray(estimates) - actual
    return np.mean(errors**2)

MSE는 실제 값을 알아야 구할 수 있습니다. 실무에서는 대개 실제 값을 모릅니다. 안다면 추정할 필요도 없을 것입니다. 하지만 이 실험에서는 실제 모평균이 3.7kg임을 아니까 표본평균의 MSE를 구할 수 있습니다.

In [16]:
mse(means, mu)

크기 10의 표본에서 표본평균으로 모평균을 추정하면 평균제곱오차는 약 0.021kg²입니다. 다음은 표본중앙값의 MSE입니다.

In [17]:
mse(medians, mu)

표본중앙값으로 모평균을 추정하면 평균제곱오차는 약 0.029kg²입니다. 이 예에서는 표본평균이 더 좋습니다. 일반적으로 자료가 정규분포에서 나왔다면, 표본평균은 MSE를 최소화한다는 의미에서 모평균의 가장 좋은 불편추정량입니다.

MSE가 작다는 것은 좋은 성질이지만, 항상 오차를 요약하는 가장 좋은 방법은 아닙니다. 우선 해석이 어렵습니다. 여기서 MSE의 단위는 kg²여서 무엇을 뜻하는지 말하기 어렵습니다.

한 가지 방법은 평균제곱오차의 제곱근(RMSE)을 사용하는 것입니다. 또 다른 방법은 오차 절댓값의 평균인 평균절대오차(MAE)를 쓰는 것입니다. 다음 함수는 추정값 시퀀스의 MAE를 계산합니다.

In [18]:
def mae(estimates, actual):
    """Mean absolute error of a sequence of estimates."""
    errors = np.asarray(estimates) - actual
    return np.mean(np.abs(errors))

표본평균의 MAE는 다음과 같습니다.

In [19]:
mae(means, mu)

표본중앙값의 MAE도 구합니다.

In [20]:
mae(medians, mu)

평균적으로 표본평균은 약 0.115kg, 표본중앙값은 약 0.137kg 어긋날 것으로 예상합니다. 적어도 이 예에서는 표본평균이 더 나은 선택인 듯합니다.

## 강건성

다른 상황을 생각해 보겠습니다. 펭귄의 체중을 재려 할 때 2%의 확률로 펭귄이 저울의 단위 버튼을 눌러, 킬로그램 대신 파운드로 체중이 기록된다고 하겠습니다. 이 오류를 알아차리지 못하면 표본에 이상치가 생깁니다.

다음 함수는 체중의 2%에 킬로그램당 약 2.2파운드라는 환산 계수를 곱해 이 상황을 시뮬레이션합니다.

In [21]:
def make_sample_with_errors(n):
    sample = np.random.normal(mu, sigma, size=n)
    factor = np.random.choice([1, 2.2], p=[0.98, 0.02], size=n)
    return sample * factor

이 오류가 분포에 미치는 영향을 보려고 큰 표본을 만들겠습니다.

In [22]:
sample = make_sample_with_errors(n=1000)

표본의 분포는 [6장의 커널 밀도 추정](/posts/statistics/think-stats-06#커널-밀도-추정)과 `Pdf` 객체로 그리겠습니다.

In [23]:
from scipy.stats import gaussian_kde
from thinkstats import Pdf

kde = gaussian_kde(sample)
domain = 0, 10
pdf = Pdf(kde, domain)
pdf.plot(label='estimated density')
decorate(xlabel="Penguin weight (kg)", ylabel="Density")

3.7kg 근처의 최빈값 외에, 측정 오류 때문에 8kg 근처에 두 번째 봉우리가 생깁니다.

앞 실험을 반복해 크기 10의 표본을 많이 만들고 각 표본의 평균을 구한 뒤, 표본평균들의 평균을 구하겠습니다.

In [24]:
means = [np.mean(make_sample_with_errors(n=10)) for i in range(10001)]
np.mean(means)

측정 오류 때문에 표본평균은 평균적으로 3.7kg보다 높아집니다.

이번에는 표본중앙값으로 같은 실험을 하겠습니다.

In [25]:
medians = [np.median(make_sample_with_errors(n=10)) for i in range(10001)]
np.mean(medians)

표본중앙값들의 평균도 3.7kg보다 높지만, 어긋난 정도는 훨씬 작습니다. MSE를 비교하면 표본중앙값이 상당히 더 정확합니다.

In [26]:
mse(means, mu), mse(medians, mu)

측정값이 정규분포에서 나온다면, 표본평균은 모평균의 불편추정량 가운데 MSE를 최소화합니다. 하지만 이 상황은 그 가정을 어기므로 더 이상 그렇지 않습니다. 표본중앙값은 이상치에 덜 민감해 편향과 MSE가 더 작습니다. 이상치나 비슷한 가정 위반을 잘 견디는 추정량을 **강건한**(robust) 추정량이라고 합니다.

## 분산 추정하기

다른 예로 펭귄 체중의 분산을 추정하려 한다고 하겠습니다. [1장의 요약 통계량](/posts/statistics/think-stats-01#요약-통계량)에서 표본의 분산을 구하는 두 방법을 보았습니다. 차이를 나중에 설명하겠다고 했는데, 이제 그 차례입니다.

두 방법이 있는 이유는 하나는 모분산의 편향된 추정량이고 다른 하나는 불편추정량이기 때문입니다. 다음 함수는 편차 제곱의 합을 `n`으로 나누는 편향된 추정량을 구합니다.

In [27]:
def biased_var(xs):
    # Compute variance with n in the denominator
    n = len(xs)
    deviations = xs - np.mean(xs)
    return np.sum(deviations**2) / n

확인하려고 크기 10의 표본을 많이 만들고 각 표본의 편향된 분산을 구한 다음, 그 분산들의 평균을 구하겠습니다.

In [28]:
biased_vars = [biased_var(make_sample(n=10)) for i in range(10001)]
np.mean(biased_vars)

결과는 약 0.19입니다. 실제 모분산이 약 0.21임을 알고 있으므로, 이 방식의 표본분산은 평균적으로 너무 낮습니다. 편향이 있음을 확인할 수 있습니다.

In [29]:
actual_var = sigma**2
actual_var

다음 함수는 편차 제곱의 합을 `n-1`로 나누는 불편추정량을 구합니다.

In [30]:
def unbiased_var(xs):
    # Compute variance with n-1 in the denominator
    n = len(xs)
    deviations = xs - np.mean(xs)
    return np.sum(deviations**2) / (n - 1)

표본을 많이 만들고 각각의 불편분산을 구해 확인할 수 있습니다.

In [31]:
unbiased_vars = [unbiased_var(make_sample(n=10)) for i in range(10001)]
np.mean(unbiased_vars)

불편 표본분산의 평균은 실제 값에 아주 가깝습니다. 불편추정량이라면 기대하는 결과입니다.

표본 크기가 10이면 두 추정량의 차이는 약 10%여서 무시하기 어려울 수 있습니다. 크기가 100이면 1%뿐이므로 실무에서 중요하지 않을 만큼 작을 수 있습니다.

In [32]:
n = 10
1 - (n - 1) / n

In [33]:
n = 100
1 - (n - 1) / n

## 표집분포

지금까지는 모수를 아는 정규분포에서 펭귄 체중이 나온다고 가정해 시뮬레이션 자료를 사용했습니다. 이제 실제 자료에서는 어떤 일이 일어나는지 보겠습니다.

2007~2010년에 남극 팔머 기지의 연구자들이 지역 개체군의 펭귄 342마리를 측정하고 체중을 쟀습니다. 이 자료는 공개되어 있습니다. 내려받는 방법은 이 장의 노트북에 있습니다.

준비 셀에서는 Allison Horst의 저장소와 같은 자료 파일을 불러옵니다.

자료 출처: Horst AM, Hill AP, Gorman KB (2020). *palmerpenguins: Palmer Archipelago (Antarctica) penguin data*. R package version 0.1.0. [자료 안내](https://allisonhorst.github.io/palmerpenguins/), [DOI: 10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218).

이 자료는 다음 논문으로 이어진 연구에서 수집했습니다. Gorman KB, Williams TD, Fraser WR (2014). Ecological sexual dimorphism and environmental variability within a community of Antarctic penguins (genus Pygoscelis). *PLoS ONE* 9(3), e90081. [DOI: 10.1371/journal.pone.0090081](https://doi.org/10.1371/journal.pone.0090081).

In [34]:
download(
    "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/c19a904462482430170bfe2c718775ddb7dbb885/inst/extdata/penguins_raw.csv"
)

pandas로 자료를 읽을 수 있습니다.

In [35]:
penguins = pd.read_csv("penguins_raw.csv").dropna(subset=["Body Mass (g)"])
penguins.shape

자료에는 세 종의 펭귄이 있습니다.

In [36]:
penguins["Species"].value_counts()

첫 예제에서는 턱끈펭귄(Chinstrap)만 선택하겠습니다.

In [37]:
chinstrap = penguins.query('Species.str.startswith("Chinstrap")')

추정한 PDF를 그릴 때 다음 함수를 사용하겠습니다.

In [38]:
def plot_kde(sample, name="estimated density", **options):
    kde = gaussian_kde(sample)
    m, s = np.mean(sample), np.std(sample)
    plt.axvline(m, color="gray", ls=":")

    domain = m - 4 * s, m + 4 * s
    pdf = Pdf(kde, domain, name)
    pdf.plot(**options)

다음은 킬로그램 단위 턱끈펭귄 체중의 분포입니다. 수직 점선은 표본평균입니다.

In [39]:
weights = chinstrap["Body Mass (g)"] / 1000
plot_kde(weights, "weights")
decorate(xlabel="Penguin weight (kg)", ylabel="Density")

표본평균은 약 3.7kg입니다.

In [40]:
sample_mean = np.mean(weights)
sample_mean

모평균을 추정해 달라는 요청을 받으면 3.7kg은 합리적인 답입니다. 하지만 이 추정은 얼마나 정밀할까요?

한 가지 방법은 평균의 **표집분포**(sampling distribution)를 구하는 것입니다. 표본을 다르게 뽑을 때 추정한 평균이 얼마나 변하는지 보여 줍니다. 실제 모평균과 모표준편차를 안다면 표본추출 과정을 모형으로 만들어 표집분포를 구할 수 있습니다. 하지만 실제 모평균을 안다면 추정할 필요가 없을 것입니다!

다행히 표집분포를 간단히 근사하는 **재표집**(resampling)이라는 방법이 있습니다. 표본으로 모집단의 모형을 만들고, 그 모형으로 표본을 뽑는 과정을 시뮬레이션하는 것이 핵심입니다.

더 구체적으로는 **모수적 재표집**(parametric resampling)을 사용하겠습니다. 표본으로 모집단의 모수를 추정한 뒤, 이론분포에서 새 표본을 생성하는 방법입니다.

다음 함수는 정규분포로 이 과정을 구현합니다. 새 표본의 크기는 원래 표본과 같습니다.

In [41]:
def resample(sample):
    # Generate a sample from a normal distribution
    m, s = np.mean(sample), np.std(sample)
    return np.random.normal(m, s, len(sample))

다음 반복문은 `resample`로 표본을 많이 만들고 각각의 평균을 구합니다.

In [42]:
sample_means = [np.mean(resample(weights)) for i in range(1001)]

이 표본평균들의 분포는 다음과 같습니다.

In [43]:
plot_kde(sample_means, "sample means")
decorate(xlabel="Sample mean of weight (kg)", ylabel="Density")

이 결과는 표본평균의 표집분포를 근사합니다. 모집단 모형이 정확하다는 가정 아래, 같은 크기의 표본을 여러 번 모으면 표본평균이 얼마나 달라질지 보여 줍니다.

대략 살펴보면 같은 크기의 다른 표본에서 평균이 3.55처럼 낮게, 또는 3.9처럼 높게 나올 수도 있음을 알 수 있습니다.

## 표준오차

표집분포의 너비를 수치로 나타내는 한 방법은 그 분포의 표준편차를 구하는 것입니다. 이 값을 **표준오차**(standard error)라고 합니다.

In [44]:
standard_error = np.std(sample_means)
standard_error

여기서 표준오차는 약 0.045kg입니다. 많은 표본을 모았을 때 표본평균들이 퍼져 있는 정도가 약 0.045kg이라는 뜻입니다. 이 값은 표본평균들의 표준편차입니다.

자료의 표준편차와 추정량의 표준오차는 서로 다른 변동을 나타냅니다.

- 자료의 표준편차는 개별 측정값들이 얼마나 퍼져 있는지 나타냅니다.
- 추정량의 표준오차는 표본을 다시 뽑을 때 추정값이 얼마나 달라지는지 나타냅니다.

이 자료에서 턱끈펭귄의 체중 표준편차는 약 0.38kg입니다.

In [45]:
np.std(weights)

평균 체중의 표준오차는 약 0.046kg입니다.

In [46]:
np.std(sample_means)

표준편차는 펭귄들 사이의 체중 차이를 알려줍니다. 표준오차는 추정이 얼마나 정밀한지 알려줍니다. 서로 다른 질문에 답하는 값입니다.

다만 둘 사이에는 관계가 있습니다. 같은 분포에서 독립적으로 뽑은 표본이라면, 표본 표준편차와 표본 크기로 평균의 표준오차를 다음과 같이 추정할 수 있습니다.

In [47]:
def approximate_standard_error(sample):
    n = len(sample)
    return np.std(sample) / np.sqrt(n)

In [48]:
approximate_standard_error(weights)

재표집으로 구한 값과 가깝습니다.

## 신뢰구간

표집분포를 요약하는 다른 방법은 **신뢰구간**(confidence interval)을 구하는 것입니다. 예를 들어 여기서는 근사한 표집분포의 가운데 90%가 들어가는 구간을 5백분위수와 95백분위수로 구합니다. 다음은 턱끈펭귄의 평균 체중에 대한 90% 신뢰구간입니다.

In [49]:
ci90 = np.percentile(sample_means, [5, 95])
ci90

신뢰구간을 보면 “모집단 모수의 참값이 90% 신뢰구간에 들어갈 확률은 90%”라고 해석하고 싶어집니다. 이 예에서는 턱끈펭귄의 모평균이 3.66~3.81kg에 있을 확률이 90%라고 말하는 것입니다.

**빈도주의**(frequentism)에서는 모수를 고정된 값으로 보기 때문에 위 문장과 다르게 해석합니다. 많은 통계학 책에서 이 해석이 틀렸다고 설명합니다.

제 생각에는 이런 금지가 불필요하게 엄격합니다. 합리적인 확률 관점에서는 신뢰구간이 사람들이 기대하는 뜻을 갖습니다. 참값이 90% 신뢰구간 안에 들어갈 확률이 90%라는 뜻입니다.

> 번역자 주: 바로 앞 문단은 저자의 해석적 견해입니다. 빈도주의에서 90%는 가정에 맞게 표본추출과 구간 계산을 반복할 때의 장기 포함률을 뜻합니다. 이미 구한 특정 구간에 고정된 모수가 들어갈 확률이 90%라는 해석과는 구분해야 합니다. 이 장의 모수적 재표집 구간도 근사이며, 모형과 표본 크기에 따라 실제 포함률이 달라질 수 있습니다. [NIST의 신뢰구간 해석](https://www.itl.nist.gov/div898/handbook/eda/section3/eda352.htm)을 참고하세요.

어떤 해석을 택하든 신뢰구간은 표집, 즉 모집단의 일부만 측정해서 생기는 변동만 수치로 나타냅니다. 표집분포는 다른 오차 원인을 반영하지 않습니다. 특히 표집편향과 측정 오류는 다음 절에서 다루겠습니다.

## 오차의 원인

남극 펭귄 대신 거주하는 도시의 성인 여성 평균 체중을 알고 싶다고 하겠습니다. 대표성 있는 여성 표본을 무작위로 뽑아 직접 체중을 재기는 어렵습니다.

간단한 대안은 “전화 표집”입니다. 전화번호부에서 번호를 무작위로 골라 전화하고, 성인 여성을 바꿔 달라고 해 체중을 묻는 것입니다. 하지만 여기에는 분명한 문제가 있습니다.

전화번호가 등록된 사람만 조사하므로 전화가 없는 사람과 비공개 번호를 쓰는 사람을 제외합니다. 전자는 평균보다 소득이 낮고 후자는 높을 수도 있습니다. 낮에 집 전화로 전화하면 직장이 있는 사람을 조사할 가능성이 작아집니다. 전화를 받은 사람만 조사하면 전화 회선을 함께 쓰는 사람이 뽑힐 가능성도 작아집니다.

소득, 고용, 가구 크기 같은 요인이 체중과 관련 있다면 조사 결과에 영향을 줍니다. 그럴 가능성은 충분합니다. 이 문제는 표본을 모으는 과정의 성질이므로 **표집편향**(sampling bias)이라고 합니다.

이 표집 과정은 표집편향의 한 종류인 자기선택에도 취약합니다. 질문에 답하기를 거절하는 사람이 있고, 거절하는 경향이 체중과 관련 있다면 결과에 영향을 줍니다.

마지막으로 직접 체중을 재지 않고 물어보면 정확하지 않을 수 있습니다. 협조적인 응답자도 실제 체중을 밝히기 불편해 올리거나 내려서 답할 수 있습니다. 모든 응답자가 협조적인 것도 아닙니다. 이런 부정확함은 **측정 오류**(measurement error)의 예입니다.

추정값을 보고할 때 표준오차나 신뢰구간을 함께 제시하면 표집 때문에 생기는 변동을 수치로 나타낼 수 있습니다. 하지만 이것은 오차 원인의 하나일 뿐이며, 가장 큰 오차도 아닐 때가 많다는 점을 기억하세요.

## 용어 정리

- **모평균**(population mean): 일부에서 계산한 표본평균과 달리, 모집단 전체에서 어떤 양의 실제 평균.
- **모수**(parameter): 한 분포족에서 특정한 분포를 정하는 값. 예를 들어 정규분포의 모수는 평균과 표준편차입니다.
- **추정량**(estimator): 모집단의 모수를 추정하려고 표본에서 계산하는 통계량.
- **일치성**(consistent): 표본 크기가 커질수록 추정량이 모수의 실제 값으로 수렴하는 성질.
- **불편성**(unbiased): 특정 표본 크기에서 추정값들의 평균이 모수의 실제 값과 같은 성질.
- **평균제곱오차**(mean squared error, MSE): 추정값과 실제 모수 값의 차이를 제곱한 값의 평균. 실제 값을 안다고 가정해 추정의 정확도를 평가합니다.
- **강건성**(robust): 자료에 이상치나 오류가 있거나 이론분포를 정확히 따르지 않아도 추정이 크게 흔들리지 않는 성질.
- **재표집**(resampling): 표본을 뽑는 과정을 시뮬레이션해 추정량의 표집분포를 근사하는 방법.
- **모수적 재표집**(parametric resampling): 표본으로 모집단 모수를 추정한 뒤 이론분포에서 표본추출 과정을 시뮬레이션하는 재표집 방법.
- **표집분포**(sampling distribution): 같은 모집단에서 가능한 표본들에 걸친 통계량의 분포.
- **표준오차**(standard error): 표집분포의 표준편차. 무작위 표집 때문에 추정값이 달라지는 정도를 나타내며, 측정 오류나 대표성 없는 표집은 포함하지 않습니다.
- **신뢰구간**(confidence interval): 모수를 추정하기 위한 구간. 신뢰수준은 같은 절차를 반복할 때 구간이 참값을 포함하는 장기 비율을 뜻합니다. 이 장에서는 재표집 분포의 백분위수로 근사 구간을 구합니다.
- **표집편향**(sampling bias): 표본 수집 방식 때문에 표본이 모집단을 대표하지 못하게 되는 문제.
- **측정 오류**(measurement error): 관측·측정·기록의 부정확함.

## 연습문제

### 연습문제 8.1

재표집 방법의 장점 하나는 다른 통계량으로 쉽게 확장할 수 있다는 것입니다. 이 장에서는 펭귄 체중의 표본평균을 구하고 재표집으로 평균의 표집분포를 근사했습니다. 이번에는 표준편차에 같은 일을 해 보겠습니다.

턱끈펭귄 체중의 표본표준편차를 구하세요. `resample`로 표준편차의 표집분포를 근사하세요. 그 표집분포로 추정값의 표준오차와 90% 신뢰구간을 구하세요.

In [50]:
# Solution goes here

In [51]:
# Solution goes here

In [52]:
# Solution goes here

In [53]:
# Solution goes here

In [54]:
# Solution goes here

### 연습문제 8.2

행동위험요인감시체계(BRFSS) 자료에는 미국 성인 표본이 직접 보고한 키와 체중이 있습니다. 이 자료로 성인 남성의 평균 키를 추정하세요. `resample`로 표집분포를 근사하고 90% 신뢰구간을 구하세요.

표본이 아주 커서 신뢰구간이 아주 좁습니다. 무작위 표집으로 인한 변동이 작다는 뜻입니다. 하지만 다른 오차 원인이 더 클 수 있습니다. 어떤 다른 원인이 결과에 영향을 준다고 생각하나요?

다음 셀들은 자료를 내려받아 `DataFrame`으로 읽고 남성 응답자의 키를 선택합니다.

In [55]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/CDBRFS08.ASC.gz")

In [56]:
from thinkstats import read_brfss

brfss = read_brfss()

In [57]:
male = brfss.query("sex == 1")
heights = male["htm3"]
heights.describe()

In [58]:
# Solution goes here

In [59]:
# Solution goes here

In [60]:
# Solution goes here

In [61]:
# Solution goes here

In [62]:
# Solution goes here

### 연습문제 8.3

[6장](/posts/statistics/think-stats-06#지수분포의-pdf)에서 보았듯, 축구나 하키의 득점 사이 시간은 지수분포를 따르는 경향이 있습니다. 득점 간격의 표본을 관측했다고 하겠습니다. 표본이 지수분포에서 나왔다고 가정할 때 실제 분포의 평균을 어떻게 추정할 수 있을까요? 표본평균이나 표본중앙값을 생각해 볼 수 있습니다. 이 추정량들의 일치성과 불편성을 살펴보겠습니다. 실험에서는 실제 평균 득점 간격이 10분이라고 가정합니다.

In [63]:
actual_mean = 10

다음 함수는 이 평균과 주어진 표본 크기의 지수분포 표본을 만듭니다.

In [64]:
def make_exponential(n):
    return np.random.exponential(actual_mean, size=n)

이 함수로 여러 크기의 표본을 만들고 각각의 평균을 구하세요. `n`이 커지면 표본평균이 실제 평균으로 수렴하나요?

이어서 여러 크기의 표본을 만들고 각각의 중앙값을 구하세요. 표본중앙값은 실제 중앙값으로 수렴하나요?

평균이 주어진 지수분포의 실제 중앙값은 다음과 같습니다.

In [65]:
actual_median = np.log(2) * actual_mean
actual_median

In [66]:
# Solution goes here

In [67]:
# Solution goes here

이번에는 크기 10의 표본을 많이 만들어, 표본평균이 모평균의 불편추정량인지 확인하세요.

In [68]:
# Solution goes here

마지막으로 표본중앙값이 모집단 중앙값의 불편추정량인지 확인하세요.

In [69]:
# Solution goes here

In [70]:
# Solution goes here

### 연습문제 8.4

이 장에서는 편향된 분산 추정량에 실제로 편향이 있음을 확인했고, 불편추정량의 평균이 실제 모분산과 일치한다는 것도 확인했습니다. 이번에는 표준편차를 살펴보겠습니다.

모집단의 표준편차를 추정하려면 다음처럼 편향된 분산 추정량이나 불편분산 추정량에 제곱근을 취할 수 있습니다.

In [71]:
def biased_std(sample):
    # Square root of the biased estimator of variance
    var = biased_var(sample)
    return np.sqrt(var)

In [72]:
def unbiased_std(sample):
    # Square root of the unbiased estimator of variance
    var = unbiased_var(sample)
    return np.sqrt(var)

`make_sample`로 평균 3.7, 표준편차 0.46인 정규분포에서 크기 10의 표본을 많이 만드세요. 두 추정량 중 표준편차의 불편추정량이 있는지 확인하세요.

In [73]:
# Here's an example using `make_sample`

mu, sigma = 3.7, 0.46
make_sample(n=10)

In [74]:
# Solution goes here

In [75]:
# Solution goes here

In [76]:
# Solution goes here

### 연습문제 8.5

이 문제는 독일 탱크 문제에 바탕을 둡니다. 제2차 세계대전 때 런던 주재 미국 대사관의 경제전 부서가 실제로 수행한 분석을 단순화한 문제입니다.

연합군 첩보원으로서 독일이 탱크를 몇 대 만들었는지 추정하는 임무를 맡았다고 하겠습니다. 자료는 노획한 탱크 `k`대에서 얻은 일련번호입니다.

독일에 1부터 `N`까지 번호를 붙인 탱크 `N`대가 있고, 이 범위의 모든 탱크가 노획될 가능성이 같다고 가정하면 `N`을 다음과 같이 추정할 수 있습니다.

In [77]:
def estimate_tanks(sample):
    m = np.max(sample)
    k = len(sample)
    return m + (m - k) / k

예로 `N`이 122라고 하겠습니다.

In [78]:
N = 122
tanks = np.arange(1, N + 1)

다음 함수로 탱크 `k`대의 무작위 표본을 만들 수 있습니다.

In [79]:
def sample_tanks(k):
    return np.random.choice(tanks, replace=False, size=k)

예를 들어 보겠습니다.

In [80]:
np.random.seed(17)

In [81]:
sample = sample_tanks(5)
sample

이 표본에서 구한 추정값은 다음과 같습니다.

In [82]:
estimate_tanks(sample)

이 추정량에 편향이 있는지 확인하세요.

In [83]:
# Solution goes here

자세한 내용은 [독일 탱크 문제 문서](https://en.wikipedia.org/wiki/German_tank_problem)와 Ruggles·Brodie의 「An Empirical Approach to Economic Intelligence in World War II」, *Journal of the American Statistical Association*, 1947년 3월을 참고하세요. [논문 사본](https://web.archive.org/web/20170123132042/https://www.cia.gov/library/readingroom/docs/CIA-RDP79R01001A001300010013-3.pdf)을 볼 수 있습니다.

추정량의 원리는 [이 영상](https://www.youtube.com/watch?v=WLCwMRJBhuI)에서도 설명합니다.

### 연습문제 8.6

여러 운동 경기, 특히 농구에서 선수와 팬들은 “핫 핸드”라는 현상을 믿습니다. 연속으로 몇 번 성공한 선수는 다음에도 성공할 가능성이 높고, 연속으로 실패한 선수는 다음에도 실패할 가능성이 높다는 생각입니다.

유명한 논문 하나는 프로 농구 경기의 성공·실패 시퀀스로 핫 핸드가 실제 현상인지 착각인지 검증하는 방법을 제안했습니다. 저자들은 선수마다 전체 슛 성공 확률과 세 번 연속 성공 뒤의 조건부 성공 확률을 계산했습니다. 선수 아홉 명 중 여덟 명에게서 세 번 성공한 뒤의 성공률이 더 낮았습니다. 이를 비롯한 결과를 바탕으로 연속한 슛 결과 사이에 양의 상관이 있다는 근거가 없다고 결론 내렸습니다. 이후 수십 년 동안 많은 사람이 핫 핸드가 반박되었다고 믿었습니다.

하지만 이 결론에는 적어도 일부 통계 오류가 있습니다. 2018년 논문은 첫 논문에서 사용한 통계량, 즉 세 번 성공한 뒤의 성공률에 편향이 있음을 보였습니다. 모든 슛의 성공 확률이 정확히 0.5이고 슛들 사이에 상관이 전혀 없어도, 각 시퀀스에서 구한 세 번 성공 뒤의 성공률을 평균하면 0.5보다 작게 나옵니다.

> 번역자 주: 독립인 슛에서 다음 슛의 실제 성공 확률은 여전히 0.5입니다. 편향이 있는 대상은 유한한 시퀀스마다 계산한 조건부 성공률 추정값의 평균입니다.

이유가 직관적이지 않아 오류가 오래 발견되지 않았고, 여기서 설명하려 하지는 않겠습니다. 대신 이 장의 방법으로 확인할 수 있습니다. 다음 함수로 0과 1이 각각 확률 0.5로 나오고 상관이 없는 시퀀스를 만들겠습니다.

In [84]:
def make_hits_and_misses(n):
    # Generate a random sequence of 0s and 1s
    return np.random.choice([0, 1], size=n)

이 장의 노트북에는 세 번 성공, 즉 1이 세 개 연속된 부분 시퀀스를 모두 찾아 그다음 원소를 반환하는 함수가 있습니다.

In [85]:
import numpy as np


def get_successors(seq, target_sum=3):
    """Get the successors of each subsequence that sums to a target value.

    Parameters:
    seq (array-like): Sequence of 1s and 0s.
    target_sum (int): The target sum of the subsequence. Default is 3.

    Returns:
    np.ndarray: Array of successors to subsequences that sum to `target_sum`.
    """
    # Check if the input sequence is too short
    if len(seq) < 3:
        return np.array([])

    # Compute the sum of each subsequence of length 3
    kernel = [1, 1, 1]
    corr = np.correlate(seq, kernel, mode="valid")

    # Find the indices where the subsequence sums to the target value
    indices = np.nonzero(corr == target_sum)[0]

    # Remove cases where the subsequence is at the end of the sequence
    indices = indices[indices < len(seq) - 3]

    # Find the successors of each valid subsequence
    successors = seq[indices + 3] if len(indices) > 0 else np.array([])

    return successors

길이 100의 시퀀스를 많이 만들고, 각 시퀀스에서 세 번 연속 성공 바로 뒤의 슛을 찾으세요. 그 슛들 중 성공의 비율을 구하세요. 힌트: 세 번 연속 성공이 없는 시퀀스에서는 함수가 빈 시퀀스를 반환하므로 이를 처리해야 합니다.

이 시뮬레이션을 많이 반복하면 성공 비율의 평균은 얼마인가요? 시퀀스 길이를 늘리거나 줄이면 어떻게 달라지나요?

In [86]:
# Solution goes here

In [87]:
# Solution goes here

In [88]:
# Solution goes here

In [89]:
# Solution goes here

In [90]:
# Solution goes here

첫 논문은 Gilovich, T., Vallone, R., & Tversky, A. (1985). The hot hand in basketball: On the misperception of random sequences. *Cognitive Psychology*, 17(3), 295–314입니다.

통계 오류를 보인 논문은 Miller, J. B., & Sanjurjo, A. (2018). Surprised by the hot hand fallacy? A truth in the law of small numbers. *Econometrica*, 86(6), 2019–2047입니다.

[첫 논문](https://www.joelvelasco.net/teaching/122/Gilo.Vallone.Tversky.pdf)과 [두 번째 논문](https://marketing.wharton.upenn.edu/wp-content/uploads/2018/11/Paper-Joshua-Miller.pdf)을 볼 수 있습니다. 주제의 개요와 오류 설명은 [이 영상](https://www.youtube.com/watch?v=CR5vT44ZMK8)에서도 다룹니다.

[Think Stats: Exploratory Data Analysis in Python, 3rd Edition](https://allendowney.github.io/ThinkStats/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

코드 라이선스: [MIT](https://mit-license.org/)

본문·그림과 한국어 번역 라이선스: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)